# Step 6c — Multi-task Learning (C2)

**RESS 2025 — GAN-Conformal-RUL**

Adds a stage-classification head to the shared encoder, trained jointly with the RUL head:
$L = L_\text{RUL} + \lambda\, L_\text{stage}$. Both heads train on all windows; the piecewise target
makes healthy RUL learnable, so no masking is needed, and healthy windows inform the encoder through
the stage head.

**Question:** does the auxiliary stage task improve near-failure RUL (accuracy and/or coverage)
versus the single-task baseline? Swept over $\lambda \in \{0.3, 0.5, 1.0\}$, multi-seed, reported
through the standard grid so it is directly comparable to the baseline and +GAN results.

## 1. Setup

In [ ]:
import os, sys, shutil
os.chdir('/content')
REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH); sys.path.insert(0, f'{REPO_PATH}/src')
from google.colab import drive
drive.mount('/content/drive')
import torch
print('CUDA:', torch.cuda.is_available())

Cloning into '/content/RESS_2025_GAN_Conformal_RUL'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 403 (delta 130), reused 5 (delta 5), pack-reused 201 (from 1)
Receiving objects: 100% (403/403), 8.09 MiB | 20.87 MiB/s, done.
Resolving deltas: 100% (223/223), done.
Mounted at /content/drive
CUDA: True


In [ ]:
import numpy as np, pandas as pd
from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import prepare_all_folds, build_folds, WINDOW_SIZE
from model import ModelConfig, RULTrainer, evaluate_grid, average_grids
from multitask import MTConfig, MultiTaskTrainer

SEEDS = [1, 2, 3, 4, 5]
LAMBDAS = [0.3, 0.5, 1.0]

[health_indicator] v2.0-guarded-fpt loaded  (FPT: 5 consecutive x max(mu+3sigma, mu*1.10))


## 2. Data (piecewise, log_clip)

In [ ]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_ROOT
all_data = XJTUSYLoader(DATA_ROOT).load_all()
results = HealthIndicatorPipeline(fpt_consecutive=5,
                                  fpt_min_relative_rise=0.20).process_all(all_data, verbose=False)
fold_data = prepare_all_folds(results, scaling_method='log_clip',
                              rul_target='piecewise', verbose=False)
folds = build_folds(results)

def test_bearing_ids(fold):
    bids = []
    for bid in fold['test']:
        n = results[bid]['features'].shape[0]
        bids.extend([bid] * max(0, n - WINDOW_SIZE + 1))
    return np.array(bids)
print('data ready.')


=== Condition 1: 35.0 Hz, 12.0 kN ===
Loading Bearing1_1 (1-1): 123 recordings, failure mode: Outer race
  -> Shape: (123, 32768, 2), Memory: 32.2 MB
Loading Bearing1_2 (1-2): 161 recordings, failure mode: Outer race
  -> Shape: (161, 32768, 2), Memory: 42.2 MB
Loading Bearing1_3 (1-3): 158 recordings, failure mode: Outer race
  -> Shape: (158, 32768, 2), Memory: 41.4 MB
Loading Bearing1_4 (1-4): 122 recordings, failure mode: Cage
  -> Shape: (122, 32768, 2), Memory: 32.0 MB
Loading Bearing1_5 (1-5): 52 recordings, failure mode: Outer race + Ball
  -> Shape: (52, 32768, 2), Memory: 13.6 MB

=== Condition 2: 37.5 Hz, 11.0 kN ===
Loading Bearing2_1 (2-1): 491 recordings, failure mode: Inner race
  -> Shape: (491, 32768, 2), Memory: 128.7 MB
Loading Bearing2_2 (2-2): 161 recordings, failure mode: Outer race
  -> Shape: (161, 32768, 2), Memory: 42.2 MB
Loading Bearing2_3 (2-3): 533 recordings, failure mode: Cage
  -> Shape: (533, 32768, 2), Memory: 139.7 MB
Loading Bearing2_4 (2-4): 42 re

## 3. Baseline anchor (single-task, multi-seed) — for comparison

In [ ]:
def eval_baseline():
    per_seed = []
    for seed in SEEDS:
        cfg = ModelConfig(epochs=100, patience=25, lr=5e-4, seed=seed)
        grids = []
        for d, f in zip(fold_data, folds):
            tr = RULTrainer(cfg)
            tr.fit(d['X_train'], d['y_rul_train'], d['X_val'], d['y_rul_val'],
                   stage_train=d['y_stage_train'], stage_val=d['y_stage_val'],
                   mask_healthy=False, verbose=False)
            yp = tr.predict(d['X_test'])
            grids.append(evaluate_grid(d['y_rul_test'], yp,
                                       d['y_stage_test'], test_bearing_ids(f)))
        per_seed.append(average_grids(grids))
        print(f'  baseline seed {seed} done')
    return per_seed

base_seeds = eval_baseline()

  baseline seed 1 done
  baseline seed 2 done
  baseline seed 3 done
  baseline seed 4 done
  baseline seed 5 done


## 4. Multi-task λ sweep (multi-seed)

In [ ]:
def eval_multitask(lam):
    per_seed = []
    for seed in SEEDS:
        cfg = MTConfig(epochs=100, patience=25, lr=5e-4, seed=seed, lambda_stage=lam)
        grids, stage_accs = [], []
        for d, f in zip(fold_data, folds):
            tr = MultiTaskTrainer(cfg)
            tr.fit(d['X_train'], d['y_rul_train'], d['y_stage_train'],
                   d['X_val'], d['y_rul_val'], verbose=False)
            yp = tr.predict_rul(d['X_test'])
            grids.append(evaluate_grid(d['y_rul_test'], yp,
                                       d['y_stage_test'], test_bearing_ids(f)))
            stage_accs.append(tr.stage_accuracy(d['X_test'], d['y_stage_test']))
        g = average_grids(grids); g['_stage_acc'] = float(np.mean(stage_accs))
        per_seed.append(g)
        print(f'  λ={lam} seed {seed} done  (stage acc {np.mean(stage_accs):.3f})')
    return per_seed

mt_seeds = {}
for lam in LAMBDAS:
    print(f'\n=== multi-task λ={lam} ===')
    mt_seeds[lam] = eval_multitask(lam)


=== multi-task λ=0.3 ===
  λ=0.3 seed 1 done  (stage acc 0.677)
  λ=0.3 seed 2 done  (stage acc 0.676)
  λ=0.3 seed 3 done  (stage acc 0.695)
  λ=0.3 seed 4 done  (stage acc 0.643)
  λ=0.3 seed 5 done  (stage acc 0.647)

=== multi-task λ=0.5 ===
  λ=0.5 seed 1 done  (stage acc 0.664)
  λ=0.5 seed 2 done  (stage acc 0.683)
  λ=0.5 seed 3 done  (stage acc 0.696)
  λ=0.5 seed 4 done  (stage acc 0.649)
  λ=0.5 seed 5 done  (stage acc 0.683)

=== multi-task λ=1.0 ===
  λ=1.0 seed 1 done  (stage acc 0.677)
  λ=1.0 seed 2 done  (stage acc 0.673)
  λ=1.0 seed 3 done  (stage acc 0.756)
  λ=1.0 seed 4 done  (stage acc 0.669)
  λ=1.0 seed 5 done  (stage acc 0.681)


## 5. Comparison — does multi-task help near-failure?

In [ ]:
def ms(seeds, sub, metric='per_bearing'):
    vals = [s[sub][metric] for s in seeds]
    return np.mean(vals), np.std(vals)

print(f"{'config':16s} {'nearfail RMSE':>18s} {'overall RMSE':>18s} {'stage acc':>10s}")
print('-'*66)
bm, bs = ms(base_seeds, 'nearfail'); om, os_ = ms(base_seeds, 'overall')
print(f"{'baseline':16s} {bm:8.4f} ± {bs:.4f}   {om:8.4f} ± {os_:.4f}   {'—':>10s}")
for lam in LAMBDAS:
    nm, ns = ms(mt_seeds[lam], 'nearfail'); vm, vs = ms(mt_seeds[lam], 'overall')
    sa = np.mean([s['_stage_acc'] for s in mt_seeds[lam]])
    print(f"{'MT λ='+str(lam):16s} {nm:8.4f} ± {ns:.4f}   {vm:8.4f} ± {vs:.4f}   {sa:10.3f}")
print(f"\nbaseline near-failure RMSE {bm:.4f} ± {bs:.4f} — a MT config must beat this by > the std to count.")

config                nearfail RMSE       overall RMSE  stage acc
------------------------------------------------------------------
baseline           0.2438 ± 0.0180     0.4598 ± 0.2187            —
MT λ=0.3           0.2165 ± 0.0178     0.4131 ± 0.1751        0.668
MT λ=0.5           0.2308 ± 0.0313     0.4188 ± 0.1719        0.675
MT λ=1.0           0.2188 ± 0.0175     0.4084 ± 0.1794        0.691

baseline near-failure RMSE 0.2438 ± 0.0180 — a MT config must beat this by > the std to count.


## Reading this

- **Stage accuracy** confirms the auxiliary head actually learned (should be well above chance 0.33;
  the stages are fairly separable, so expect high accuracy).
- **Near-failure RMSE** is the test of C2: does multi-task beat the single-task baseline by more than
  the seed std? Given C1 was a coverage story not an accuracy one, MT may also be neutral on RMSE —
  in which case its value (if any) would show in conformal coverage, tested next by feeding the best
  MT model through the Step 7 conformal pipeline.
- If no λ beats baseline beyond noise, C2 is a controlled negative on point accuracy, reported honestly;
  the contribution then rests on whether the shared representation improves coverage.